### --- SECTION 0: Setup ---

In [1]:
import pandas as pd
import numpy as np

### Create a "messy" dataset to simulate real-world conditions

In [2]:
raw_data = {
    'transaction_id': [1001, 1002, 1003, 1004, 1005, 1001, 1006, 1007, 1008, 1009],
    'date': ['2024-05-01', '2024-05-02', 'May 3rd, 2024', '2024-05-04', None, '2024-05-01', '2024-05-06', '2024-05-07', '2024-05-08', '2024-05-09'],
    'store_branch': [' North ', 'south', 'North', 'EAST', 'west', ' North ', 'south', 'North', 'West', None],
    'product': ['Laptop', 'Mouse', 'Monitor', 'Keyboard', 'Laptop', 'Laptop', 'Mouse', 'Monitor', 'Tablet', 'Tablet'],
    'price_usd': ['$1,200', '25.50', '300', '$85.00', '1200', '$1,200', '25.50', None, '450', '450'],
    'units_sold': [1, 2, 1, 5, 1, 1, 2, 1, None, 3]
}

df = pd.DataFrame(raw_data)
print("--- RAW DATA PREVIEW ---")
display(df.head())

--- RAW DATA PREVIEW ---


,transaction_id,date,store_branch,product,price_usd,units_sold
0,1001,2024-05-01,North,Laptop,"$1,200",1.0
1,1002,2024-05-02,south,Mouse,25.50,2.0
2,1003,"May 3rd, 2024",North,Monitor,300,1.0
3,1004,2024-05-04,EAST,Keyboard,$85.00,5.0
4,1005,None,west,Laptop,1200,1.0


##- SECTION 1: Inspection (Skill: .info & .isna) ---

### --- SECTION 1: Inspection (Skill: .info & .isna) ---

In [3]:
print("\n--- INITIAL INSPECTION ---")
df.info()
print("\nMissing values per column:")
print(df.isna().sum())


--- INITIAL INSPECTION ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   transaction_id  10 non-null     int64  
 1   date            9 non-null      object 
 2   store_branch    9 non-null      object 
 3   product         10 non-null     object 
 4   price_usd       9 non-null      object 
 5   units_sold      9 non-null      float64
dtypes: float64(1), int64(1), object(4)
memory usage: 612.0+ bytes

Missing values per column:
transaction_id    0
date              1
store_branch      1
product           0
price_usd         1
units_sold        1
dtype: int64


### --- SECTION 2: Removing Duplicates (Skill: .drop_duplicates) ---

In [4]:
before = len(df)
df = df.drop_duplicates()
print(f"\nDuplicates removed: {before - len(df)}")


Duplicates removed: 1


### --- SECTION 3: Fixing Types & Formatting (Skill: pd.to_numeric, .str, pd.to_datetime) ---

In [5]:
# 3.1 Standardize Branch Names (Strip whitespace and proper case)
df['store_branch'] = df['store_branch'].str.strip().str.title()
df['store_branch'] = df['store_branch'].fillna('Unknown')

# 3.2 Clean Price Column (Remove symbols and convert to float)
df['price_usd'] = df['price_usd'].str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['price_usd'] = pd.to_numeric(df['price_usd'], errors='coerce')

# 3.3 Convert Dates
df['date'] = pd.to_datetime(df['date'], errors='coerce')
# Drop rows where date is missing as it's critical for time-series
df = df.dropna(subset=['date'])

### --- SECTION 4: Handling Missing Values (Skill: .fillna) ---

In [6]:
# Fill missing units with 1 (sensible default)
df['units_sold'] = df['units_sold'].fillna(1).astype(int)

# Fill missing prices with the median price for that product
df['price_usd'] = df['price_usd'].fillna(df.groupby('product')['price_usd'].transform('median'))

### --- SECTION 5: Feature Engineering (Skill: .dt, pd.cut) ---

In [7]:
# Derive Revenue
df['total_revenue'] = df['price_usd'] * df['units_sold']

# Date Features
df['day_name'] = df['date'].dt.day_name()
df['is_weekend'] = df['date'].dt.weekday >= 5

# Segmentation (Binning)
df['order_size'] = pd.cut(df['total_revenue'], 
                          bins=[0, 100, 1000, 10000], 
                          labels=['Small', 'Medium', 'Large'])

### --- SECTION 6: Final Output ---

In [8]:
print("\n--- CLEANED DATA PREVIEW ---")
display(df.head())


--- CLEANED DATA PREVIEW ---


,transaction_id,date,store_branch,product,price_usd,units_sold,total_revenue,day_name,is_weekend,order_size
0,1001,2024-05-01,North,Laptop,1200.0,1,1200.0,Wednesday,False,Large
1,1002,2024-05-02,South,Mouse,25.5,2,51.0,Thursday,False,Small
3,1004,2024-05-04,East,Keyboard,85.0,5,425.0,Saturday,True,Medium
6,1006,2024-05-06,South,Mouse,25.5,2,51.0,Monday,False,Small
7,1007,2024-05-07,North,Monitor,NaN,1,NaN,Tuesday,False,NaN


In [9]:
# Save for next steps
df.to_csv('./cleaned_retail_sales.csv', index=False)
print("\nPipeline complete. Cleaned file saved.")


Pipeline complete. Cleaned file saved.
